# Fine-tuning DataMate · BF16 LoRA · SFT v2

Chạy tiếp adapter `runs/lora/qwen35_sft_4xl40s/adapter` trên **3.665 mẫu train, đủ 1 epoch**, một GPU.
Bộ dữ liệu: `data/fine_tuning_data_20261114_v2` (212 mẫu dev gốc; lấy **64 mẫu cố định**, seed 3407).
**Đánh giá mỗi 100 optimizer step; lưu checkpoint mỗi 100 step; tắt early stopping.** Giữ đánh giá đầu lượt như công thức cũ;
Trainer có thể đánh giá/lưu thêm ở bước cuối. Adapter xuất ra là checkpoint có dev loss thấp nhất.

Thiết lập: Qwen3.5-9B gốc BF16, LoRA r16/alpha32, lr 5e-5, max length 16.384,
micro-batch 1 × accumulation 16, khoảng 230 optimizer step. Không tải model hoặc đổi dữ liệu trong notebook.

**Chuẩn bị môi trường một lần**, đặt base model và adapter ở đúng đường dẫn:

```bash
python3 -m venv .venv-train
.venv-train/bin/python -m pip install -r training/requirements_bf16.txt
.venv-train/bin/python -m pip install --no-deps fla-core==0.6.0
.venv-train/bin/python -m pip install nbformat ipykernel pandas matplotlib
```

**Một lệnh từ thư mục gốc repo** (stdout hiển thị trực tiếp; lỗi trả mã khác 0):

```bash
.venv-train/bin/python -u -c "import nbformat; n=nbformat.read('notebooks/02_train_lora.ipynb',4); exec('\n\n'.join(c.source for c in n.cells if c.cell_type=='code'))"
```

Trong Jupyter: chọn môi trường có các gói trên rồi **Run All**. Kiểm tra CPU/tokenizer trước bằng cách đặt
`DATAMATE_MODE=check` trước lệnh; train mặc định là `train`. Tiếp tục đúng checkpoint bị gián đoạn: `RESUME=1` trước lệnh.
Có thể đặt `DATA_DIR`, `MODEL_DIR`, `INIT_ADAPTER`, `OUT`, `TRAIN_PYTHON`, `TOKENIZER_DIR` qua biến môi trường.

In [ ]:
import os
import sys
from pathlib import Path

candidates = [Path(os.environ.get("DATAMATE_REPO", ".")).expanduser().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in candidates if (p / "training/train_lora_bf16.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook trong checkout datamate hoặc đặt DATAMATE_REPO=/path/to/datamate_clone")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import hashlib
import json
import math
import random
import subprocess
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from IPython.display import Image, display

IN_JUPYTER = type(get_ipython()).__name__ == "ZMQInteractiveShell"


def path(value):
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p


def show_table(frame):
    display(frame) if IN_JUPYTER else print(frame.to_string(index=False), flush=True)


def show_figure(fig, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(dest, dpi=150, bbox_inches="tight")
    if IN_JUPYTER:
        display(Image(filename=str(dest)))
    plt.close(fig)
    print(f"Figure: {dest}", flush=True)


def file_sha256(filename):
    h = hashlib.sha256()
    with open(filename, "rb") as handle:
        while chunk := handle.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

## Cấu hình đường dẫn

Đường dẫn tương đối tính từ repo; chuyển sang máy train chỉ cần giữ cấu trúc repo hoặc đặt các biến dưới đây.

In [ ]:
MODE = os.environ.get("DATAMATE_MODE", "train")
DATA_DIR = os.environ.get("DATA_DIR", "data/fine_tuning_data_20261114_v2")
MODEL_DIR = os.environ.get("MODEL_DIR", "models/Qwen3.5-9B")
INIT_ADAPTER = os.environ.get("INIT_ADAPTER", "runs/lora/qwen35_sft_4xl40s/adapter")
OUT = os.environ.get("OUT", "runs/lora/qwen35_sft_h100_v2")
RESUME = os.environ.get("RESUME", "0") == "1"
CUDA_DEVICE = os.environ.get("CUDA_VISIBLE_DEVICES", "0")
TRAIN_PYTHON = os.environ.get("TRAIN_PYTHON", str(ROOT / ".venv-train/bin/python") if (ROOT / ".venv-train/bin/python").exists() else sys.executable)
TOKENIZER_DIR = os.environ.get("TOKENIZER_DIR", "")

data_dir, model_dir, init_adapter, out = map(path, (DATA_DIR, MODEL_DIR, INIT_ADAPTER, OUT))
train_python = str(path(TRAIN_PYTHON))
print(f"Repo: {ROOT}\nData: {data_dir}\nModel: {model_dir}\nInit: {init_adapter}\nOutput: {out}\nPython: {train_python}\nMODE={MODE}; RESUME={RESUME}", flush=True)

## Dữ liệu và số liệu đo

Mỗi lượt đều kiểm checksum, số dòng, metadata split/role và train/dev trùng câu hỏi/shop,
rồi chạy tokenizer thật ở **max_len=16384**. Chỉ bắt đầu train khi giữ đủ 3.665 train và 212 dev gốc.
Part B có role `btc_train`; part C dev giữ nhãn nguồn `synthetic_train`, nên xác định split bằng file và `metadata.split`.
Các biến thể cùng question_id trong train được giữ nguyên; row id phải duy nhất.
Trainer lấy 64 dev cố định và tiếp tục kiểm tra identity khi resume. Kiểm tra kernel và bộ nhớ thất bại sẽ dừng.

Loss lấy từ log của Trainer, giữ nguyên giá trị được ghi; loss chỉ tính trên completion, prompt được mask.
Train loss là loss của các batch được ghi, dev loss là trung bình trên 64 mẫu cố định; đây không phải điểm đúng SQL.
Bảng cuối ghi rõ `train_loss_mean_logged` là trung bình các train loss đã log.

**ETA là ước tính**, không phải thời gian bảo đảm: trung bình 20 optimizer step gần nhất × số bước còn lại,
cộng thời gian đánh giá và checkpoint còn lại từ các lần đã đo. Lịch còn lại lấy từ `DefaultFlowCallback`
của chính phiên bản Transformers đang chạy; khi resume chỉ đếm phần chưa chạy trong phiên hiện tại.
Nếu chưa đo được một pha thì hiển thị `chưa đo`, không gán thời gian giả. CUDA được synchronize khi đo step.
ETA bao gồm phần train/eval/checkpoint, chưa bao gồm xuất adapter cuối; tổng thời gian thực đo khi kết thúc gồm cả kiểm tra và preflight.

`notebook_metrics.jsonl` giữ metrics, thời gian, ETA và bộ nhớ GPU từng sự kiện;
`notebook_stdout.log` giữ toàn bộ stdout; `training_metrics.csv` và `training_curves.png` được tạo sau khi hoàn tất.

In [ ]:
def audit_data(data_dir):
    """Read the supplied splits; never edit source rows or the bundle manifest."""
    manifest = json.loads((data_dir / "sft_manifest.json").read_text())
    for line in (data_dir / "SHA256SUMS").read_text().splitlines():
        if line.strip():
            expected, relative = line.split(maxsplit=1)
            filename = data_dir / relative.lstrip("*")
            if file_sha256(filename) != expected:
                raise ValueError(f"Checksum không khớp: {filename}")
    records, questions, shops, counts = [], {}, {}, {}
    allowed_roles = {"train": {"synthetic_train", "btc_train"}, "dev": {"synthetic_dev", "synthetic_train"}}
    for split, expected_rows in (("train", 3665), ("dev", 212)):
        filename = data_dir / f"{split}.jsonl"
        spec = manifest["splits"][split]
        if "sha256:" + file_sha256(filename) != spec["file_hash"]:
            raise ValueError(f"Không khớp sft_manifest.json: {filename}")
        questions[split], shops[split], row_ids = set(), set(), set()
        count = 0
        with filename.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                row = json.loads(line)
                meta = row["metadata"]
                if meta.get("dataset_role") not in allowed_roles[split] or meta.get("split") != split or meta.get("training_eligible") is not True:
                    raise ValueError(f"Sai role/training_eligible: {split}, {row.get('id')}")
                if [m["role"] for m in row["prompt_messages"]] != ["system", "user"] or not row["completion"]:
                    raise ValueError(f"Sai prompt/completion: {row.get('id')}")
                if row["id"] in row_ids:
                    raise ValueError(f"Trùng row id trong {split}: {row['id']}")
                row_ids.add(row["id"])
                questions[split].add(meta["question_id"])
                shops[split].add(row["business_id"])
                records.append({"split": split, "id": row["id"], "question_id": meta["question_id"], "shop": row["business_id"],
                                "part": meta.get("part", "không ghi"), "dataset_role": meta["dataset_role"], "answer_type": meta.get("answer_type", "không ghi"),
                                "route": meta.get("route", "không ghi"), "thinking": bool(row.get("thinking_mode")),
                                "question": meta.get("question", ""), "completion": row["completion"],
                                "prompt_chars": sum(len(m["content"]) for m in row["prompt_messages"]),
                                "completion_chars": len(row["completion"]), "weak_group": meta.get("weak_group", "không ghi")})
                count += 1
        if count != expected_rows or count != spec["rows"]:
            raise ValueError(f"{split}: {count} mẫu, yêu cầu {expected_rows}")
        counts[split] = count
    if questions["train"] & questions["dev"] or shops["train"] & shops["dev"]:
        raise ValueError("Train/dev trùng question_id hoặc shop")
    print(f"Checksum OK | {counts['train']:,} train / {counts['dev']} dev | không trùng câu hỏi/shop", flush=True)
    return pd.DataFrame(records)

In [ ]:
TRAINER_PAYLOAD = r'''
import copy
import json
import math
import os
import sys
import time
import uuid
from collections import deque
from datetime import datetime, timezone
from pathlib import Path

import torch
from transformers import DefaultFlowCallback, TrainerCallback, TrainerControl
from training import train_lora_bf16 as T


def sync_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def finite(value):
    if isinstance(value, dict):
        return {k: finite(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [finite(v) for v in value]
    if hasattr(value, "item"):
        value = value.item()
    return str(value) if isinstance(value, float) and not math.isfinite(value) else value


def duration(seconds):
    seconds = max(0, int(seconds))
    return f"{seconds // 3600:02d}:{seconds // 60 % 60:02d}:{seconds % 60:02d}"


class LiveMetrics(TrainerCallback):
    def __init__(self, out):
        self.out = Path(out)
        self.session = uuid.uuid4().hex[:12]
        self.started = time.perf_counter()
        self.steps = deque(maxlen=20)
        self.phase_times = {"eval": [], "checkpoint": []}
        self.targets = {"eval": set(), "checkpoint": set()}
        self.completed = {"eval": set(), "checkpoint": set()}
        self.state = None
        self.step_started = None

    def estimate(self):
        remaining = max(0, self.state.max_steps - self.state.global_step)
        missing = []
        train = sum(self.steps) / len(self.steps) * remaining if self.steps else (0.0 if not remaining else None)
        if train is None:
            missing.append("train")
        result = {"train_seconds": train}
        for phase in ("eval", "checkpoint"):
            pending = len(self.targets[phase] - self.completed[phase])
            samples = self.phase_times[phase]
            result[f"{phase}_seconds"] = sum(samples) / len(samples) * pending if samples else (0.0 if not pending else None)
            if result[f"{phase}_seconds"] is None:
                missing.append(phase)
        result["seconds"] = sum(v for v in result.values() if v is not None) if train is not None else None
        result["unmeasured"] = missing
        return result

    def emit(self, event, **values):
        eta = self.estimate()
        record = {"event": event, "session": self.session, "utc": datetime.now(timezone.utc).isoformat(),
                  "step": self.state.global_step, "max_steps": self.state.max_steps,
                  "elapsed_seconds": time.perf_counter() - self.started, "eta": eta, **values}
        if torch.cuda.is_available():
            record.update(gpu_allocated_gib=torch.cuda.memory_allocated() / 2**30,
                          gpu_reserved_gib=torch.cuda.memory_reserved() / 2**30,
                          gpu_peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30)
        record = finite(record)
        with (self.out / "notebook_metrics.jsonl").open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
        details = " ".join(f"{k}={record[k]}" for k in ("loss", "eval_loss", "learning_rate", "grad_norm", "epoch", "step_seconds", "phase_seconds") if k in record)
        missing = ", chưa đo " + "/".join(eta["unmeasured"]) if eta["unmeasured"] else ""
        if eta["seconds"] is None:
            eta_text = "ETA: chờ đo optimizer step đầu tiên"
        else:
            label = "ETA phần đã đo" if eta["unmeasured"] else "ETA ước tính"
            eta_text = f"{label} {duration(eta['seconds'])}{missing}"
        print(f"[{event}] step {record['step']}/{record['max_steps']} | {details} | elapsed {duration(record['elapsed_seconds'])} | {eta_text}", flush=True)

    def on_train_begin(self, args, state, control, **kwargs):
        self.state = state
        self.started = time.perf_counter()
        simulated = copy.copy(state)
        flow = DefaultFlowCallback()
        for step in range(state.global_step + 1, state.max_steps + 1):
            simulated.global_step = step
            plan = flow.on_step_end(args, simulated, TrainerControl())
            if plan.should_evaluate:
                self.targets["eval"].add(step)
            if plan.should_save:
                self.targets["checkpoint"].add(step)
        if args.eval_on_start:
            self.targets["eval"].add(state.global_step)
        self.emit("start", eval_steps_planned=sorted(self.targets["eval"]), checkpoint_steps_planned=sorted(self.targets["checkpoint"]))

    def on_step_begin(self, args, state, control, **kwargs):
        sync_gpu()
        self.step_started = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        sync_gpu()
        self.steps.append(time.perf_counter() - self.step_started)
        self.emit("step_time", step_seconds=self.steps[-1])

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            self.emit("metrics", **logs)

    def finish_phase(self, phase, seconds):
        self.phase_times[phase].append(seconds)
        self.completed[phase].add(self.state.global_step)
        self.emit(phase, phase_seconds=seconds)


def check_estimator():
    from types import SimpleNamespace
    m = LiveMetrics(".")
    m.state = SimpleNamespace(global_step=100, max_steps=230)
    m.targets = {"eval": {100, 200, 230}, "checkpoint": {100, 200, 230}}
    m.completed = {"eval": {100}, "checkpoint": {100}}
    assert set(m.estimate()["unmeasured"]) == {"train", "eval", "checkpoint"}
    assert m.estimate()["seconds"] is None
    m.steps.extend([1.0, 3.0])
    m.phase_times = {"eval": [10.0], "checkpoint": [5.0]}
    assert m.estimate()["seconds"] == 290.0  # 130 steps x 2s + 2 evals x 10s + 2 saves x 5s
    m.state.global_step = 200  # resume: only the remaining 30 optimizer steps
    m.completed = {"eval": {100, 200}, "checkpoint": {100, 200}}
    assert m.estimate()["seconds"] == 75.0
    m.state.global_step = 230
    m.completed = {k: set(v) for k, v in m.targets.items()}
    assert m.estimate()["seconds"] == 0.0 and not m.estimate()["unmeasured"]


check_estimator()
print("ETA self-check PASS (unknown phases / remaining steps / resume / complete)", flush=True)
original_make_trainer = T.make_trainer


def instrumented_trainer(*args, **kwargs):
    trainer = original_make_trainer(*args, **kwargs)
    if len(trainer.train_dataset) != 3665 or len(trainer.eval_dataset) != 64:
        raise RuntimeError("Dừng: lượt train phải có đúng 3.665 train và 64 dev")
    monitor = LiveMetrics(trainer.args.output_dir)
    for callback in list(trainer.callback_handler.callbacks):
        if type(callback).__module__ == T.__name__ and type(callback).__name__ == "Log":
            trainer.remove_callback(type(callback))
    trainer.add_callback(monitor)

    def timed(phase, function):
        def call(*call_args, **call_kwargs):
            sync_gpu()
            started = time.perf_counter()
            result = function(*call_args, **call_kwargs)
            sync_gpu()
            monitor.finish_phase(phase, time.perf_counter() - started)
            return result
        return call

    trainer.evaluate = timed("eval", trainer.evaluate)
    trainer._save_checkpoint = timed("checkpoint", trainer._save_checkpoint)
    return trainer


T.make_trainer = instrumented_trainer
T.main(sys.argv[1:])
'''

In [ ]:
def trainer_args(destination, dev=True):
    args = ["--train", str(data_dir / "train.jsonl"), "--model", str(model_dir), "--out", str(destination),
            "--init-adapter", str(init_adapter), "--max-len", "16384", "--r", "16", "--alpha", "32", "--dropout", "0",
            "--lr", "5e-5", "--epochs", "1", "--micro-batch", "1", "--grad-accum", "16", "--warmup-ratio", "0.03",
            "--scheduler", "cosine", "--weight-decay", "0", "--max-grad-norm", "1", "--seed", "3407", "--attn", "sdpa",
            "--max-steps", "-1", "--logging-steps", "1", "--eval-steps", "100", "--dev-max-rows", "64", "--early-stop", "0",
            "--ce-chunk", "1024", "--save-steps", "100", "--save-total-limit", "3", "--report-to", "none"]
    if dev:
        args += ["--dev", str(data_dir / "dev.jsonl")]
    return args


def run_stage(label, command, log_file, env):
    print(f"\n{label}\nLog: {log_file}", flush=True)
    log_file.parent.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    with log_file.open("a", encoding="utf-8") as log:
        log.write(f"\n--- {label} {time.strftime('%Y-%m-%d %H:%M:%S')} ---\n")
        log.flush()
        with subprocess.Popen([str(x) for x in command], cwd=ROOT, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1, errors="replace") as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                status = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    elapsed = time.perf_counter() - started
    if status:
        raise RuntimeError(f"{label} thất bại, exit={status}; xem {log_file}")
    print(f"{label}: {elapsed:.1f} giây", flush=True)
    return elapsed


def render_results(workflow_seconds):
    state = json.loads((out / "trainer_state.json").read_text())
    manifest = json.loads((out / "run_manifest.json").read_text())
    if (manifest["identity"]["rows_train"], manifest["identity"]["rows_dev"]) != (3665, 64):
        raise RuntimeError("Run manifest không đúng số mẫu yêu cầu")
    if state["global_step"] != state["max_steps"] or not math.isclose(state["epoch"], 1.0, abs_tol=1e-6):
        raise RuntimeError("Lượt train chưa hoàn thành đủ 1 epoch")
    history = pd.DataFrame(state["log_history"])
    history.to_csv(out / "training_metrics.csv", index=False)
    events = pd.read_json(out / "notebook_metrics.jsonl", lines=True)
    show_table(pd.DataFrame([{
        "train_rows": manifest["identity"]["rows_train"], "dev_rows": manifest["identity"]["rows_dev"],
        "epoch": state["epoch"], "optimizer_steps": state["global_step"],
        "train_loss_mean_logged": manifest["train_loss"], "best_dev_loss": state.get("best_metric"),
        "workflow_seconds_this_session": round(workflow_seconds, 3),
        "trainer_seconds_this_session": manifest["seconds"], "peak_gpu_allocated_GiB": manifest["peak_gpu_gib"],
        "peak_gpu_reserved_GiB": manifest["peak_gpu_reserved_gib"], "best_checkpoint": state.get("best_model_checkpoint")
    }]))
    train = history.dropna(subset=["loss"])
    dev = history.dropna(subset=["eval_loss"])
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
    axes[0, 0].plot(train["step"], train["loss"], lw=1, alpha=0.7, label="Train: logged batch loss")
    axes[0, 0].plot(dev["step"], dev["eval_loss"], "o-", label="Dev: fixed 64 rows")
    axes[0, 0].set(title="Completion-only loss", ylabel="Cross-entropy")
    axes[0, 0].legend()
    axes[0, 1].plot(train["step"], train["learning_rate"])
    axes[0, 1].set(title="Learning rate", ylabel="Learning rate")
    axes[1, 0].plot(train["step"], train["grad_norm"])
    axes[1, 0].set(title="Gradient norm", ylabel="Norm")
    measured = events.loc[events["event"].eq("step_time")]
    if "session" in events:
        measured = measured.loc[measured["session"].eq(events["session"].iloc[-1])]
    if not measured.empty:
        axes[1, 1].plot(measured["step"], measured["step_seconds"], ".-", ms=3)
    axes[1, 1].set(title="Optimizer step time (current session)", ylabel="Seconds (CUDA synchronized)")
    for ax in axes.flat:
        ax.set_xlabel("Optimizer step")
        ax.grid(alpha=0.2)
    show_figure(fig, out / "training_curves.png")
    print(f"Adapter: {out / 'adapter'}\nCSV: {out / 'training_metrics.csv'}\nLive metrics / ETA: {out / 'notebook_metrics.jsonl'}\n"
          f"Full stdout: {out / 'notebook_stdout.log'}", flush=True)
    return manifest

In [ ]:
def run_finetuning():
    """One call: source audit -> real tokenizer -> kernel -> memory preflight -> full training -> report."""
    if MODE not in {"train", "check"}:
        raise ValueError("MODE phải là train hoặc check")
    started = time.perf_counter()
    if os.environ.get("WORLD_SIZE", "1") != "1":
        raise RuntimeError("Notebook này chạy trực tiếp trên 1 GPU, không dùng torchrun")
    if MODE == "train":
        for filename in (model_dir / "config.json", init_adapter / "adapter_config.json", init_adapter / "adapter_model.safetensors"):
            if not filename.is_file() or filename.stat().st_size < (1000000 if filename.suffix == ".safetensors" else 1):
                raise FileNotFoundError(f"Thiếu file local hoặc còn là LFS pointer: {filename}. Đặt MODEL_DIR / INIT_ADAPTER đúng đường dẫn.")
        existing = list(out.glob("checkpoint-*"))
        if existing and not RESUME:
            raise RuntimeError(f"{out} đã có checkpoint; dùng RESUME=True cho chính run này hoặc đổi OUT")
        if RESUME and not existing:
            raise RuntimeError("RESUME=True nhưng OUT chưa có checkpoint")
        if not RESUME and (out / "run_manifest.json").exists():
            raise RuntimeError("OUT đã có kết quả; chọn OUT mới")
    audited = audit_data(data_dir)
    show_table(audited.groupby("split").agg(rows=("id", "size"), shops=("shop", "nunique"), thinking=("thinking", "sum")).reset_index())
    del audited
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(CUDA_DEVICE), "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1",
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    env.pop("DATAMATE_TRAIN_ALLOW_CPU", None)
    tokenizer = path(TOKENIZER_DIR) if TOKENIZER_DIR else model_dir if (model_dir / "tokenizer_config.json").exists() else ROOT / "adapters/sql_rft_v3"
    report_dir = out.with_name(out.name + "_data_check")
    # Each execution validates again. An old report is never a fallback for a failing check.
    run_stage("1/4 · Tokenizer + masks + max length 16.384", [train_python, "-u", "training/train_lora_bf16.py",
              *trainer_args(report_dir), "--tokenizer", str(tokenizer), "--validate-only"], report_dir / "validate.log", env)
    report = json.loads((report_dir / "data_report.json").read_text())
    for split, expected in (("train", 3665), ("dev", 212)):
        if report[split]["rows_read"] != expected or report[split]["rows_kept"] != expected or report[split]["rejected"]:
            raise RuntimeError(f"{split} không giữ đủ {expected} mẫu ở max_len=16384")
    print("Giữ đủ 3.665 train; đánh giá cùng 64/212 dev (seed 3407); 1 epoch, khoảng 230 optimizer step.", flush=True)
    if MODE == "check":
        print("MODE=check hoàn tất; không nạp weights, không chạy GPU/training.", flush=True)
        return report
    gpu_check = (
        "import torch; from importlib.util import find_spec; "
        "assert find_spec('peft') and find_spec('accelerate'), 'Cài training/requirements_bf16.txt'; "
        "assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'Cần GPU BF16'; "
        "p=torch.cuda.get_device_properties(0); assert p.total_memory/2**30 >= 39, 'Cần GPU >=40 GB và memory preflight'; "
        "print('GPU:', p.name, 'GiB:', round(p.total_memory/2**30,2), 'torch:', torch.__version__)"
    )
    run_stage("GPU / BF16 / training dependencies", [train_python, "-u", "-c", gpu_check], out / "notebook_stdout.log", env)
    run_stage("2/4 · FLA kernel", [train_python, "-u", "training/check_fla.py", "--T", "4096"], out / "kernel.log", env)
    if "fla check PASS" not in (out / "kernel.log").read_text().split("--- 2/4")[-1]:
        raise RuntimeError("FLA chưa PASS; dừng trước khi train")
    if not RESUME:
        import tempfile
        out.parent.mkdir(parents=True, exist_ok=True)
        preflight = Path(tempfile.mkdtemp(prefix=out.name + "_preflight_", dir=out.parent))
        run_stage("3/4 · Memory preflight: longest sequence + longest completion",
                  [train_python, "-u", "training/train_lora_bf16.py", *trainer_args(preflight, dev=False),
                   "--grad-accum", "1", "--max-steps", "2", "--longest-first", "--limit-rows", "0"], preflight / "preflight.log", env)
        memory = json.loads((preflight / "run_manifest.json").read_text())
        use = memory["peak_gpu_reserved_gib"] / memory["gpu_total_gib"]
        print(f"Preflight reserved {memory['peak_gpu_reserved_gib']} / {memory['gpu_total_gib']} GiB ({use:.1%})", flush=True)
        if use > 0.95:
            raise RuntimeError("Preflight sát giới hạn bộ nhớ (>95%); dừng trước khi train")
    else:
        print("3/4 · Resume đúng run: giữ kiểm tra identity của trainer, bỏ qua memory preflight.", flush=True)
    run_stage("4/4 · BF16 LoRA · 3.665 train · 64 dev · 1 epoch",
              [train_python, "-u", "-c", TRAINER_PAYLOAD, *trainer_args(out), *(["--resume"] if RESUME else [])], out / "notebook_stdout.log", env)
    return render_results(time.perf_counter() - started)

## Chạy

Một lần gọi chạy hết các bước và tự xuất bảng/đồ thị. `MODE="check"` chỉ kiểm dữ liệu và tokenizer, không nạp model.

In [ ]:
result = run_finetuning()